# 6장 1강: CV 기반 하이퍼파라미터 튜닝과 모델 탐색

## 실습 목표

이번 실습에서는 **Ames Housing** 데이터를 이용하여 하이퍼파라미터 탐색과 교차검증을 결합해 최적 모델을 찾습니다.

- GridSearchCV와 RandomizedSearchCV의 탐색 방식과 비용 차이를 비교합니다.
- Optuna의 TPE 탐색을 이용해 초기 탐색 이후 이전 trial 결과를 활용하는 탐색을 수행합니다.
- 교차검증 평균 점수와 표준편차를 함께 확인합니다.
- 탐색 시간까지 고려하여 최종 후보를 선정합니다.
- 처음부터 분리해 둔 hold-out test set은 최종 모델을 정한 뒤 한 번만 사용합니다.
- 최종 모델과 메타데이터를 저장합니다.

> 이 실습에서는 6장 1강 교안에서 다룬 Grid Search, Random Search, Optuna, CV 평균·표준편차, 탐색 비용, hold-out test, 모델 저장 내용만 사용합니다.


## 실습 환경 / 데이터

- Python
- pandas
- scikit-learn
- scipy
- Optuna
- joblib
- 데이터: `Ames_Housing.csv`

### 분석 목표

주택 특성을 이용하여 `SalePrice`를 예측하는 회귀 문제입니다.

### 사용할 특성

- `OverallQual`
- `GrLivArea`
- `GarageCars`
- `GarageArea`
- `TotalBsmtSF`
- `1stFlrSF`
- `2ndFlrSF`
- `FullBath`
- `TotRmsAbvGrd`
- `YearBuilt`
- `YearRemodAdd`
- `Fireplaces`

예측 대상은 `SalePrice`입니다.

## 실습 준비

하이퍼파라미터 탐색에 Test 데이터를 사용하면 안 됩니다.

따라서 먼저 전체 데이터를 **Train / hold-out Test**로 분리하고,

- 하이퍼파라미터 탐색과 교차검증 → Train 데이터만 사용
- 최종 성능 확인 → 모든 선택이 끝난 뒤 hold-out Test를 한 번만 사용

하는 흐름으로 진행합니다.

In [1]:
import os
import json
import time
import joblib
import optuna
import pandas as pd

from datetime import datetime
from scipy.stats import randint

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    RandomizedSearchCV,
    cross_val_score
)
from sklearn.ensemble import RandomForestRegressor

house = pd.read_csv("Ames_Housing.csv")

features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "GarageArea",
    "TotalBsmtSF",
    "1stFlrSF",
    "2ndFlrSF",
    "FullBath",
    "TotRmsAbvGrd",
    "YearBuilt",
    "YearRemodAdd",
    "Fireplaces"
]

X = house[features]
y = house["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Train 크기:", X_train.shape)
print("Hold-out Test 크기:", X_test.shape)

c:\practical_machine_learning_assignment_seungwon\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Train 크기: (1168, 12)
Hold-out Test 크기: (292, 12)


---

# 필수 1. Grid Search와 Random Search 비교

## 배경

GridSearchCV는 지정한 모든 조합을 빠짐없이 평가하고, RandomizedSearchCV는 정해진 횟수만큼 조합을 무작위로 선택합니다.

이번 실습에서는 같은 RandomForestRegressor를 대상으로 두 탐색 전략을 비교합니다.

## 문제 1. GridSearchCV를 수행하세요.

### 요구사항

1. `RandomForestRegressor(random_state=42, n_jobs=-1)`를 사용합니다.
2. 다음 후보를 사용합니다.

```python
grid_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 8, 12]
}
```

3. `cv=5`를 사용합니다.
4. `scoring="r2"`를 사용합니다.
5. 탐색 시간을 측정합니다.
6. `best_params_`와 `best_score_`를 출력합니다.
7. `cv_results_`에서 최적 조합의 `mean_test_score`, `std_test_score`를 확인합니다.

### 확인 포인트

- 총 9개 조합을 모두 평가하는가?
- 5-Fold이므로 각 조합이 5번 평가되는 구조를 이해했는가?
- 평균 점수뿐 아니라 표준편차도 확인했는가?

In [2]:
# TODO
# GridSearchCV를 수행하세요.

grid_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 8, 12]
}

# 1. 모델 정의 및 시간 측정 시작
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

start_time = time.time()

# 2~4. GridSearchCV 설정
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=grid_params,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

# 학습 수행
grid_search.fit(X_train, y_train)

# 5. 탐색 시간 계산 및 출력
elapsed_time = time.time() - start_time
print(f"GridSearchCV 탐색 완료 시간: {elapsed_time:.2f}초")

# 6. best_params_ 및 best_score_ 출력
print(f"\n[최적 파라미터 및 R2 점수]")
print(f"Best Parameters: {grid_search.best_params_}")
print(f"Best Score (Mean CV R2): {grid_search.best_score_:.4f}")

# 7. cv_results_에서 최적 조합의 mean_test_score 및 std_test_score 확인
best_index = grid_search.best_index_
best_mean_score = grid_search.cv_results_["mean_test_score"][best_index]
best_std_score = grid_search.cv_results_["std_test_score"][best_index]

print(f"\n[최적 파라미터 조합의 CV 결과 상세]")
print(f"Mean Test Score (R2): {best_mean_score:.4f}")
print(f"Std Test Score: {best_std_score:.4f}")

GridSearchCV 탐색 완료 시간: 5.16초

[최적 파라미터 및 R2 점수]
Best Parameters: {'max_depth': 12, 'n_estimators': 200}
Best Score (Mean CV R2): 0.8206

[최적 파라미터 조합의 CV 결과 상세]
Mean Test Score (R2): 0.8206
Std Test Score: 0.0644


In [3]:
grid_seconds = elapsed_time

### Q1. Grid Search의 가장 큰 장점과 한계는 무엇인가요?

**답변:**  
여기에 작성하세요.
장점: 모든 하이퍼파라미터 조합 다 탐색할 수 있는 것
한계: 시간 오래걸리는 것

## 문제 2. RandomizedSearchCV를 수행하세요.

### 요구사항

1. 같은 RandomForestRegressor를 사용합니다.
2. 다음 탐색 공간을 사용합니다.

```python
random_params = {
    "n_estimators": randint(100, 301),
    "max_depth": randint(4, 13)
}
```

3. `n_iter=9`를 사용합니다.
4. `cv=5`, `scoring="r2"`를 사용합니다.
5. `random_state=42`를 사용합니다.
6. 탐색 시간을 측정합니다.
7. 최적 파라미터, CV 평균 점수, 표준편차를 확인합니다.

### 확인 포인트

- 전체 조합을 모두 보는 것이 아니라 9번만 무작위로 탐색하는가?
- `n_iter`가 탐색 예산을 직접 정한다는 점을 이해했는가?

In [4]:
# TODO
# RandomizedSearchCV를 수행하세요.

random_params = {
    "n_estimators": randint(100, 301),
    "max_depth": randint(4, 13)
}

# 1. 모델 정의 및 시간 측정 시작
rf = RandomForestRegressor(random_state=42, n_jobs=-1)

start_time = time.time()

# 2~5. RandomizedSearchCV 설정
random_search = RandomizedSearchCV(
    estimator=rf,
    param_distributions=random_params,
    n_iter=9,
    cv=5,
    scoring="r2",
    random_state=42,
    n_jobs=-1
)

# 학습 수행
random_search.fit(X_train, y_train)

# 6. 탐색 시간 계산 및 출력
elapsed_time = time.time() - start_time
print(f"RandomizedSearchCV 탐색 완료 시간: {elapsed_time:.2f}초")

# 7. 최적 파라미터, CV 평균 점수, 표준편차 확인
best_index = random_search.best_index_
best_mean_score = random_search.cv_results_["mean_test_score"][best_index]
best_std_score = random_search.cv_results_["std_test_score"][best_index]

print(f"\n[최적 파라미터 및 R2 점수]")
print(f"Best Parameters: {random_search.best_params_}")
print(f"Mean Test Score (CV R2): {best_mean_score:.4f}")
print(f"Std Test Score: {best_std_score:.4f}")

RandomizedSearchCV 탐색 완료 시간: 2.22초

[최적 파라미터 및 R2 점수]
Best Parameters: {'max_depth': 11, 'n_estimators': 230}
Mean Test Score (CV R2): 0.8210
Std Test Score: 0.0651


In [5]:
random_seconds = elapsed_time 

### Q2. Random Search가 Grid Search보다 유리할 수 있는 상황은 언제인가요?

**답변:**  
하이퍼 파라미터 조합이 너무 많을때, 빨리 문제를 해결해야 할때

---

# 필수 2. Optuna와 CV 결합

## 배경

Grid와 Random은 이전 탐색 결과와 상관없이 후보를 선택합니다.

이번 실습에서는 Optuna의 `TPESampler`를 사용합니다.  
처음 3개의 trial은 초기 탐색으로 수행하고, 이후 trial부터는 앞선 결과를 참고하여 다음 후보를 제안하도록 설정합니다.

## 문제 1. Optuna objective에 교차검증을 연결하세요.

### 요구사항

1. `objective(trial)` 함수를 만듭니다.
2. 다음 하이퍼파라미터를 탐색합니다.
   - `n_estimators`: 100 ~ 300
   - `max_depth`: 4 ~ 12
3. `RandomForestRegressor`를 사용합니다.
4. `cross_val_score(..., cv=5, scoring="r2")`를 사용합니다.
5. CV 평균 점수를 반환합니다.
6. 다음 설정의 `TPESampler`를 만듭니다.

```python
sampler = optuna.samplers.TPESampler(
    seed=42,
    n_startup_trials=3
)
```

7. `direction="maximize"`와 위 sampler를 사용해 study를 만듭니다.
8. `n_trials=9`로 탐색합니다.
9. 탐색 시간을 측정합니다.
10. `best_params`, `best_value`를 출력합니다.

### 확인 포인트

- 한 번의 시도를 trial이라고 부르는가?
- 탐색 전체를 study라고 부르는가?
- 처음 3개 trial 이후에는 이전 trial 결과를 활용하는 TPE 탐색이 이루어지는가?
- CV 평균 성능을 Optuna의 최적화 대상으로 사용했는가?


In [6]:
# TODO
# Optuna objective + CV를 구성하세요.
# TPESampler(seed=42, n_startup_trials=3)를 사용하고
# 총 9 trial을 실행하세요.

# 1~5. objective 함수 정의
def objective(trial):
    # 2. 하이퍼파라미터 탐색 범위 설정
    n_estimators = trial.suggest_int("n_estimators", 100, 300)
    max_depth = trial.suggest_int("max_depth", 4, 12)
    
    # 3. 모델 정의
    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42,
        n_jobs=-1
    )
    
    # 4. cross_val_score 계산 (cv=5, scoring="r2")
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring="r2", n_jobs=-1)
    
    # 5. CV 평균 점수 반환
    return scores.mean()

# 시간 측정 시작
start_time = time.time()

# 6. TPESampler 설정
sampler = optuna.samplers.TPESampler(
    seed=42,
    n_startup_trials=3
)

# 7. Study 생성 (direction="maximize", sampler 적용)
study = optuna.create_study(
    direction="maximize",
    sampler=sampler
)

# 8. n_trials=9 로 최적화 수행 (optuna 실행 로그 출력을 줄이려면 show_progress_bar=False 옵션 추가 가능)
study.optimize(objective, n_trials=9)

# 9. 탐색 시간 계산 및 출력
elapsed_time = time.time() - start_time
print(f"Optuna 탐색 완료 시간: {elapsed_time:.2f}초")

# 10. best_params 및 best_value 출력
print(f"\n[최적 파라미터 및 R2 점수]")
print(f"Best Parameters: {study.best_params}")
print(f"Best Value (Mean CV R2): {study.best_value:.4f}")

[I 2026-10-12 20:18:42,874] A new study created in memory with name: no-name-227555b7-cbea-4686-a0b2-26fb7a070e8d
[I 2026-10-12 20:18:43,199] Trial 0 finished with value: 0.8212542415716282 and parameters: {'n_estimators': 175, 'max_depth': 12}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-12 20:18:43,593] Trial 1 finished with value: 0.8184595590761908 and parameters: {'n_estimators': 247, 'max_depth': 9}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-12 20:18:43,794] Trial 2 finished with value: 0.7990289774524457 and parameters: {'n_estimators': 131, 'max_depth': 5}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-12 20:18:44,043] Trial 3 finished with value: 0.8201477777785264 and parameters: {'n_estimators': 109, 'max_depth': 12}. Best is trial 0 with value: 0.8212542415716282.
[I 2026-10-12 20:18:44,413] Trial 4 finished with value: 0.8205950100254196 and parameters: {'n_estimators': 204, 'max_depth': 12}. Best is trial 0 with value: 0.821254

Optuna 탐색 완료 시간: 2.91초

[최적 파라미터 및 R2 점수]
Best Parameters: {'n_estimators': 175, 'max_depth': 12}
Best Value (Mean CV R2): 0.8213


In [7]:
optuna_seconds = elapsed_time 

### Q3. Optuna가 Grid / Random과 다른 핵심은 무엇인가요?

**답변:**  
여기에 작성하세요.
r2점수가 생성된다?

## 문제 2. Optuna 최적 조합의 CV 표준편차를 확인하세요.

### 요구사항

1. `study.best_params`로 RandomForestRegressor를 만듭니다.
2. 다시 5-Fold `cross_val_score()`를 수행합니다.
3. CV 평균과 표준편차를 계산합니다.

### 확인 포인트

- 최종 후보를 볼 때 평균만 보지 않고 Fold 간 흔들림도 확인했는가?

In [8]:
# TODO
# Optuna 최적 조합의 CV 평균과 표준편차를 확인하세요.

# 1. study.best_params 기반 최적의 RandomForestRegressor 모델 정의
best_rf = RandomForestRegressor(
    **study.best_params,
    random_state=42,
    n_jobs=-1
)

# 2. 5-Fold cross_val_score 수행
cv_scores = cross_val_score(
    best_rf,
    X_train,
    y_train,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

# 3. CV 평균과 표준편차 계산 및 출력
mean_cv_score = cv_scores.mean()
std_cv_score = cv_scores.std()

print(f"[Optuna Best Model 5-Fold CV 결과]")
print(f"Mean Test Score (CV R2): {mean_cv_score:.4f}")
print(f"Std Test Score: {std_cv_score:.4f}")

[Optuna Best Model 5-Fold CV 결과]
Mean Test Score (CV R2): 0.8213
Std Test Score: 0.0643


---

# 과제 1. 최종 모델 선정과 hold-out Test 확인

## 배경

세 탐색 전략에서 각각 최적 후보를 얻었습니다.

최종 모델은 **CV 평균을 우선 기준으로 비교**하되, 성능 차이가 매우 작다면 다음 항목도 함께 확인합니다.

- CV 표준편차
- 탐색 시간

그리고 최종 후보를 정한 뒤 **hold-out Test는 딱 한 번만** 사용합니다.

> 탐색 시간은 각 탐색 도구의 내부 동작과 실행 환경에 따라 포함되는 작업이 조금 다를 수 있으므로, 이번 실습에서는 절대적인 속도 순위가 아니라 **대략적인 탐색 비용 비교**로 해석합니다.

## 요구사항

1. Grid / Random / Optuna 결과를 다음 컬럼으로 하나의 DataFrame에 정리합니다.
   - `search_method`
   - `best_params`
   - `cv_mean`
   - `cv_std`
   - `search_seconds`
2. 세 후보를 비교합니다.
3. CV 평균을 우선 기준으로 최종 후보를 선택하고, 평균이 거의 비슷하다면 CV 표준편차와 탐색 비용도 함께 고려하여 선택 이유를 설명합니다.
4. 튜닝하지 않은 기준 모델(`RandomForestRegressor(n_estimators=100, max_depth=None, random_state=42, n_jobs=-1)`)과 3번에서 선택한 튜닝 후보를 비교하세요. 필수 실습의 `cv=5`와 동일한 `KFold(n_splits=5, shuffle=False)` 분할을 두 모델에 공통 적용하고, `cross_validate(..., scoring="r2", return_train_score=True)`로 평가하세요.
   - Fold별 Train R²와 검증 R²를 `fold_comparison`으로 정리합니다.
   - Train R² 평균, 검증 R² 평균·표준편차, `Train 평균 - 검증 평균`을 `tuning_comparison`으로 정리합니다.
   - `튜닝 후 CV 평균 R² - 기준 모델 CV 평균 R²`를 계산합니다. R² 차이값으로 보고하며, 백분율 개선율로 바꾸지 않습니다.
   - 평균·변동·Train–검증 차이로 개선 여부를 설명합니다. 튜닝이 개선되지 않았다면 기준 모델 유지도 가능합니다. Test 확인 전에 최종 모델과 선택 이유를 확정합니다.
   - 최종 선택 모델을 Train 전체에 다시 학습합니다.
5. hold-out Test에서 `.score()`로 최종 R²를 딱 한 번 확인합니다.
6. 최종 선택 모델의 CV 평균과 hold-out Test R²의 차이를 계산하고, 과적합 가능성과 일반화 판단의 한계를 설명합니다. Test 결과를 본 뒤 다시 튜닝하거나 후보를 교체하지 않습니다.
7. 선택한 모델을 `model_artifacts/best_model.pkl`로 저장합니다.
8. 다음 정보를 `model_artifacts/model_meta.json`에 저장합니다.
   - 모델 이름
   - 저장 시각
   - 선택한 탐색 방법
   - 최적 파라미터
   - CV 평균
   - CV 표준편차
   - 최종 Test R²
   - 기준 모델 설정, 튜닝 후보 설정·탐색 방법
   - 튜닝 전후 CV 비교 결과와 CV 평균 R² 차이
   - 선택 모델 CV 평균과 Test R²의 차이

### 확인 포인트

- Test 데이터를 탐색 중간에 사용하지 않았는가?
- CV 평균을 우선으로 보고, 필요할 때 표준편차와 탐색 시간도 함께 고려했는가?
- 최종 모델을 정한 뒤 Test를 한 번만 확인했는가?
- 모델뿐 아니라 메타데이터도 저장했는가?

- 튜닝 전후를 동일한 Fold에서 비교하고 개선 여부를 수치로 설명했는가?
- CV의 `test_score`는 Fold 내부 검증 점수이며 hold-out Test 점수가 아님을 구분했는가?
- 작은 표준편차나 CV/Test 점수의 일치만으로 일반화가 보장된다고 단정하지 않았는가?


In [9]:
# TODO
# 세 탐색 후보를 비교하고 기준 모델과 튜닝 후보의 동일 Fold CV 결과를 정리하세요.
# 튜닝 전후 R² 차이를 계산해 최종 선택을 확정한 뒤
# 선택 모델만 hold-out Test에서 한 번 평가하고 모델·비교 메타데이터를 저장하세요.

import numpy as np
from sklearn.model_selection import KFold, cross_validate

# 세 탐색 결과를 하나의 표로 정리
gi = grid_search.best_index_
ri = random_search.best_index_

search_summary = pd.DataFrame({
    "search_method": ["GridSearchCV", "RandomizedSearchCV", "Optuna"],
    "best_params": [grid_search.best_params_, random_search.best_params_, study.best_params],
    "cv_mean": [grid_search.cv_results_["mean_test_score"][gi],
                random_search.cv_results_["mean_test_score"][ri],
                mean_cv_score],
    "cv_std": [grid_search.cv_results_["std_test_score"][gi],
               random_search.cv_results_["std_test_score"][ri],
               std_cv_score],
    "search_seconds": [grid_seconds, random_seconds, optuna_seconds],
})
display(search_summary)

# CV 평균이 가장 높은 후보를 우선 선택
# 평균 차이가 TOL 이내면 '비슷하다'고 보고 표준편차 -> 탐색 시간 순으로 선택
TOL = 0.002
close = search_summary[search_summary["cv_mean"] >= search_summary["cv_mean"].max() - TOL]
chosen = close.sort_values(["cv_std", "search_seconds"]).iloc[0]

chosen_method = chosen["search_method"]
chosen_params = {k: int(v) for k, v in chosen["best_params"].items()}

print("선택한 탐색 방법:", chosen_method)
print("선택한 파라미터:", chosen_params)
print("선택 이유: CV 평균이 가장 높거나 비슷한 후보 중 표준편차가 가장 작음 (동률이면 탐색 시간이 짧음)")
print("※ 탐색 시간은 도구별 내부 동작이 달라 대략적인 비용 비교로만 해석")

# 기준 모델과 튜닝 후보 정의
baseline_params = {"n_estimators": 100, "max_depth": None}
baseline_model = RandomForestRegressor(**baseline_params, random_state=42, n_jobs=-1)
tuned_model = RandomForestRegressor(**chosen_params, random_state=42, n_jobs=-1)

# 두 모델에 같은 분할(KFold)을 적용해 교차 검증
kf = KFold(n_splits=5, shuffle=False)
cv_base = cross_validate(baseline_model, X_train, y_train, cv=kf,
                         scoring="r2", return_train_score=True, n_jobs=-1)
cv_tuned = cross_validate(tuned_model, X_train, y_train, cv=kf,
                          scoring="r2", return_train_score=True, n_jobs=-1)

# Fold별 Train R²와 검증 R² 비교 (val_r2는 Fold 내부 검증 점수이며 hold-out Test가 아님)
fold_comparison = pd.DataFrame({
    "fold": list(range(1, 6)) * 2,
    "model": ["baseline"] * 5 + ["tuned"] * 5,
    "train_r2": np.concatenate([cv_base["train_score"], cv_tuned["train_score"]]),
    "val_r2": np.concatenate([cv_base["test_score"], cv_tuned["test_score"]]),
})
display(fold_comparison)

# 모델별 평균/표준편차/Train-검증 차이 요약
def summarize(name, cv):
    train_mean = cv["train_score"].mean()
    val_mean = cv["test_score"].mean()
    return {"model": name,
            "train_r2_mean": train_mean,
            "val_r2_mean": val_mean,
            "val_r2_std": cv["test_score"].std(),
            "train_minus_val": train_mean - val_mean}

tuning_comparison = pd.DataFrame([summarize("baseline", cv_base),
                                  summarize("tuned", cv_tuned)])
display(tuning_comparison)

# 튜닝 후 CV 평균 R² - 기준 모델 CV 평균 R² (R² 차이값)
cv_diff = cv_tuned["test_score"].mean() - cv_base["test_score"].mean()
print(f"CV 평균 R² 차이 (튜닝 - 기준): {cv_diff:+.4f}")

# Test 확인 전에 최종 모델 확정: 개선되면 튜닝 모델, 아니면 기준 모델 유지
if cv_diff > 0:
    final_name, final_model, final_cv = "RandomForestRegressor(tuned)", tuned_model, cv_tuned
    final_method, final_params = chosen_method, chosen_params
    final_reason = "튜닝 후 CV 평균 R²가 기준 모델보다 높아 튜닝 모델 선택"
else:
    final_name, final_model, final_cv = "RandomForestRegressor(baseline)", baseline_model, cv_base
    final_method, final_params = "None (baseline)", baseline_params
    final_reason = "튜닝 후 CV 평균 R²가 개선되지 않아 기준 모델 유지"
print("최종 모델:", final_name)
print("선택 이유:", final_reason)

final_cv_mean = float(final_cv["test_score"].mean())
final_cv_std = float(final_cv["test_score"].std())

# 최종 모델을 Train 전체로 다시 학습
final_model.fit(X_train, y_train)

# hold-out Test는 여기서 딱 한 번만 확인
test_r2 = float(final_model.score(X_test, y_test))
print(f"최종 Test R²: {test_r2:.4f}")

# CV 평균과 Test R²의 차이
cv_test_gap = final_cv_mean - test_r2
print(f"CV 평균 - Test R²: {cv_test_gap:+.4f}")
print("차이가 작아도 일반화가 보장되지는 않음 (Test 표본 수, 단일 분할, 사용한 피처에 한정된 결과)")
print("Test 결과를 본 뒤에는 재튜닝하거나 후보를 바꾸지 않음")

# 모델 저장
os.makedirs("model_artifacts", exist_ok=True)
joblib.dump(final_model, "model_artifacts/best_model.pkl")

# 메타데이터 저장
meta = {
    "model_name": final_name,
    "saved_at": datetime.now().isoformat(timespec="seconds"),
    "selected_search_method": final_method,
    "best_params": final_params,
    "cv_mean": final_cv_mean,
    "cv_std": final_cv_std,
    "test_r2": test_r2,
    "baseline_config": baseline_params,
    "tuned_candidate": {"search_method": chosen_method, "params": chosen_params},
    "tuning_cv_comparison": tuning_comparison.to_dict(orient="records"),
    "cv_mean_r2_diff_tuned_minus_baseline": float(cv_diff),
    "cv_mean_minus_test_r2": float(cv_test_gap),
    "selection_reason": final_reason,
}

with open("model_artifacts/model_meta.json", "w", encoding="utf-8") as f:
    json.dump(meta, f, ensure_ascii=False, indent=2)

,search_method,best_params,cv_mean,cv_std,search_seconds
0,GridSearchCV,"{'max_depth': 12, 'n_estimators': 200}",0.820618,0.064425,5.157549
1,RandomizedSearchCV,"{'max_depth': 11, 'n_estimators': 230}",0.821030,0.065086,2.217538
2,Optuna,"{'n_estimators': 175, 'max_depth': 12}",0.821254,0.064281,2.910641


선택한 탐색 방법: Optuna
선택한 파라미터: {'n_estimators': 175, 'max_depth': 12}
선택 이유: CV 평균이 가장 높거나 비슷한 후보 중 표준편차가 가장 작음 (동률이면 탐색 시간이 짧음)
※ 탐색 시간은 도구별 내부 동작이 달라 대략적인 비용 비교로만 해석


,fold,model,train_r2,val_r2
0,1,baseline,0.971728,0.862667
1,2,baseline,0.975444,0.680790
2,3,baseline,0.974969,0.834270
3,4,baseline,0.974586,0.845045
4,5,baseline,0.971663,0.872282
5,1,tuned,0.971451,0.869414
6,2,tuned,0.976382,0.696696
7,3,tuned,0.972136,0.827994
8,4,tuned,0.972621,0.842770
9,5,tuned,0.970872,0.869396


,model,train_r2_mean,val_r2_mean,val_r2_std,train_minus_val
0,baseline,0.973678,0.819011,0.070370,0.154667
1,tuned,0.972692,0.821254,0.064281,0.151438


CV 평균 R² 차이 (튜닝 - 기준): +0.0022
최종 모델: RandomForestRegressor(tuned)
선택 이유: 튜닝 후 CV 평균 R²가 기준 모델보다 높아 튜닝 모델 선택
최종 Test R²: 0.8911
CV 평균 - Test R²: -0.0698
차이가 작아도 일반화가 보장되지는 않음 (Test 표본 수, 단일 분할, 사용한 피처에 한정된 결과)
Test 결과를 본 뒤에는 재튜닝하거나 후보를 바꾸지 않음


### Q4. CV 평균이 아주 조금 높은 모델의 탐색 시간이 몇 배 더 길다면 무조건 그 모델을 선택해야 하나요?

**답변:**  
아니다. CV 평균 차이가 아주 작다면 표준편차와 탐색 시간도 함께 보고 비용 대비 이득이 작으면 더 빠르고 안정적인 모델을 선택해도 된다.

### Q5. hold-out Test를 탐색 마지막에 한 번만 사용해야 하는 이유는 무엇인가요?

**답변:**  
Test를 탐색 중에 쓰면 그 정보가 모델 선택에 스며들어 점수가 낙관적으로 나오므로 선택을 끝낸 뒤 한번만 써야 일반화 성능을 공정하게 확인할 수 있다.

### Q6. 튜닝 전후 CV 평균 R²는 얼마나 달라졌나요? Fold별 변동과 Train–검증 점수 차이를 근거로 개선 여부를 설명하세요.

**답변:**
튜닝 후 CV 평균 R²는 0.8190에서 0.8213으로 +0.0022 올랐으나 개선 폭이 매우 작고, 검증 R² 표준편차는 0.0704에서 0.0643으로, Train–검증 차이는 0.1547에서 0.1514로 소폭 줄었다. 다만 Fold 2가 두 모델 모두 0.68~0.70대로 낮아 변동의 대부분이 데이터 분할에서 나오고 Train–검증 차이도 여전히 0.15 정도로 크므로 미세한 정도의 개선이 이루어졌다고 할 수 있다.

### Q7. CV 결과와 최종 Test 결과를 어떻게 해석했나요? 과적합 가능성과 일반화 판단의 한계를 설명하세요.

**답변:**
최종 Test R²(0.8911)가 CV 평균(0.8213)보다 0.0698 높게 나와 과적합 징후는 뚜렷하지 않지만 Train R²가 0.97로 높아 과적합 경향 자체는 남아 있다. Test가 CV보다 높은 것은 Fold 2 같은 어려운 구간이 CV에만 포함되었거나 Test 분할이 비교적 쉬웠을 가능성이 있고 Test 표본이 한번의 분할뿐이라 이 차이만으로 일반화 성능을 단정할 수 없다.

---

# 실습 마무리

1. GridSearchCV와 RandomizedSearchCV의 가장 큰 탐색 방식 차이는 무엇인가요?
2. RandomizedSearchCV의 `n_iter`는 무엇을 의미하나요?
3. Optuna에서 `study`와 `trial`은 각각 무엇인가요?
4. 최종 모델을 고를 때 CV 평균 외에 무엇을 함께 봐야 하나요?
5. hold-out Test는 언제 사용해야 하나요?
6. 최종 튜닝 모델과 메타데이터를 저장하는 이유는 무엇인가요?